In [ ]:
# pip install timm torch numpy pandas

import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

CACHE_DIR = Path("outputs/cache")
RUN_DIR = Path("outputs/runs/accuracy_predictor_rgbd")

CHANNELS = "rgbd"
TEST_PART = "Plug"
TIMM_NAME = "vit_large_patch16_dinov3.lvd1689m"
IMG_SIZE = 256
BATCH_SIZE = 64
EPOCHS = 35
LR_BACKBONE = 1e-5
LR_HEAD = 5e-4
WEIGHT_DECAY = 0.05
CORR_WEIGHT = 0.3
ATTN_WIDTH = 384
ATTN_DEPTH = 9
HEAD_DROPOUT = 0.1
SEED = 1


class TransformerHead(nn.Module):
    def __init__(self, d_in, width, depth, drop, n_tokens):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(d_in), nn.Linear(d_in, width))
        self.cls = nn.Parameter(torch.zeros(1, 1, width))
        self.pos = nn.Parameter(torch.zeros(1, n_tokens + 1, width))
        nn.init.trunc_normal_(self.cls, std=0.02)
        nn.init.trunc_normal_(self.pos, std=0.02)
        enc = nn.TransformerEncoderLayer(width, 8, width * 2, drop, activation="gelu", batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(enc, depth)
        self.out = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, width // 2), nn.GELU(),
                                 nn.Dropout(drop), nn.Linear(width // 2, 1))

    def forward(self, tokens):
        tok = torch.cat([self.cls.expand(tokens.shape[0], -1, -1), self.proj(tokens)], 1)
        n = min(tok.shape[1], self.pos.shape[1])
        tok = tok[:, :n] + self.pos[:, :n]
        return self.out(self.enc(tok)[:, 0]).squeeze(1)


class AccuracyPredictor(nn.Module):
    def __init__(self, in_chans):
        super().__init__()
        self.backbone = timm.create_model(TIMM_NAME, pretrained=True, num_classes=0, global_pool="",
                                          in_chans=in_chans, img_size=IMG_SIZE)
        with torch.no_grad():
            feat = self.backbone.eval().forward_features(torch.zeros(1, in_chans, IMG_SIZE, IMG_SIZE))
        self.backbone.train()
        self.head = TransformerHead(feat.shape[2], ATTN_WIDTH, ATTN_DEPTH, HEAD_DROPOUT, feat.shape[1])

    def forward(self, x):
        return self.head(self.backbone.forward_features(x))


def prepare_batch(x_all, idx, mean, std, train):
    x = x_all[idx].permute(0, 3, 1, 2).float()
    x[:, :3] /= 255.0
    if train:
        n = x.shape[0]
        scale = torch.empty(n, device=x.device).uniform_(0.95, 1.05)
        shift = torch.empty(n, 2, device=x.device).uniform_(-0.04, 0.04)
        theta = torch.zeros(n, 2, 3, device=x.device)
        theta[:, 0, 0] = theta[:, 1, 1] = 1.0 / scale
        theta[:, :, 2] = shift
        grid = F.affine_grid(theta, x.shape, align_corners=False)
        x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)
        b = torch.empty(n, 1, 1, 1, device=x.device).uniform_(0.9, 1.1)
        c = torch.empty(n, 1, 1, 1, device=x.device).uniform_(0.9, 1.1)
        rgb = x[:, :3]
        m = rgb.mean(dim=(1, 2, 3), keepdim=True)
        x = torch.cat([(((rgb - m) * c + m) * b).clamp_(0, 1), x[:, 3:]], 1)
    if x.shape[-1] != IMG_SIZE:
        x = F.interpolate(x, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", antialias=True, align_corners=False)
    x[:, :3] = (x[:, :3] - mean) / std
    return x


@torch.no_grad()
def predict(model, x_all, mean, std):
    model.eval()
    out = []
    for s in range(0, x_all.shape[0], 256):
        idx = torch.arange(s, min(s + 256, x_all.shape[0]), device=x_all.device)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out.append(model(prepare_batch(x_all, idx, mean, std, False)).float())
    return torch.cat(out).cpu().numpy()


def metrics(meta, y, p):
    m = {
        "MAE": float(np.mean(np.abs(y - p))),
        "RMSE": float(np.sqrt(np.mean((y - p) ** 2))),
        "Pearson": float(np.corrcoef(y, p)[0, 1]),
        "Spearman": float(pd.Series(y).corr(pd.Series(p), method="spearman")),
    }
    df = pd.DataFrame({"group": meta["group"].values, "y": y, "p": p})
    sel = np.array([(g.loc[g["p"].idxmin(), "y"], g["y"].min(), g["y"].mean()) for _, g in df.groupby("group")])
    m.update({"selected_dft_um": float(sel[:, 0].mean()), "oracle_dft_um": float(sel[:, 1].mean()),
              "random_view_dft_um": float(sel[:, 2].mean())})
    return m


torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda")
RUN_DIR.mkdir(parents=True, exist_ok=True)

meta = pd.read_csv(CACHE_DIR / "meta.csv")
x = np.load(CACHE_DIR / "images_384.npy").astype(np.float16)
if "d" in CHANNELS:
    x = np.concatenate([x, np.load(CACHE_DIR / "depth_maps.npy")[..., None]], -1)
tr = np.where(meta["part"].values != TEST_PART)[0]
te = np.where(meta["part"].values == TEST_PART)[0]
meta_tr, meta_te = meta.iloc[tr].reset_index(drop=True), meta.iloc[te].reset_index(drop=True)
x_tr = torch.from_numpy(np.ascontiguousarray(x[tr])).to(device)
x_te = torch.from_numpy(np.ascontiguousarray(x[te])).to(device)

y_tr_um = meta_tr["surface_dft_um_mean"].values.astype(np.float64)
y_te_um = meta_te["surface_dft_um_mean"].values.astype(np.float64)
mu, sd = float(np.log(y_tr_um).mean()), float(np.log(y_tr_um).std())
y_tr = torch.tensor((np.log(y_tr_um) - mu) / sd, dtype=torch.float32, device=device)
to_um = lambda z: np.exp(np.asarray(z, np.float64) * sd + mu)

model = AccuracyPredictor(x.shape[-1]).to(device)
mean = torch.tensor(model.backbone.pretrained_cfg["mean"], device=device).view(1, 3, 1, 1)
std = torch.tensor(model.backbone.pretrained_cfg["std"], device=device).view(1, 3, 1, 1)
decay = [p for p in model.backbone.parameters() if p.ndim >= 2]
no_decay = [p for p in model.backbone.parameters() if p.ndim < 2]
opt = torch.optim.AdamW([
    {"params": list(model.head.parameters()), "lr": LR_HEAD, "weight_decay": 1e-4},
    {"params": decay, "lr": LR_BACKBONE, "weight_decay": WEIGHT_DECAY},
    {"params": no_decay, "lr": LR_BACKBONE, "weight_decay": 0.0},
])
steps = len(tr) // BATCH_SIZE
total = steps * EPOCHS
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: (s + 1) / steps if s < steps else 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (s - steps) / max(1, total - steps)))
)

print(f"channels={CHANNELS} train={len(tr)} test={len(te)} ({TEST_PART}) steps/epoch={steps}", flush=True)
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(len(tr), generator=torch.Generator().manual_seed(SEED + epoch))[: steps * BATCH_SIZE].to(device)
    running = 0.0
    for s in range(steps):
        idx = perm[s * BATCH_SIZE:(s + 1) * BATCH_SIZE]
        with torch.autocast("cuda", dtype=torch.bfloat16):
            pred = model(prepare_batch(x_tr, idx, mean, std, True))
        pred, target = pred.float(), y_tr[idx]
        loss = F.mse_loss(pred, target)
        if CORR_WEIGHT > 0:
            pc, tc = pred - pred.mean(), target - target.mean()
            loss = loss + CORR_WEIGHT * (1 - (pc * tc).sum() / (pc.norm() * tc.norm() + 1e-6))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
        running += loss.item()
    msg = f"epoch {epoch:3d}  loss={running / steps:.4f}  {(time.time() - t0) / 60:.1f} min"
    if epoch % 5 == 0 or epoch == EPOCHS:
        mm = metrics(meta_te, y_te_um, to_um(predict(model, x_te, mean, std)))
        msg += f"  test MAE={mm['MAE']:.1f}um Pearson={mm['Pearson']:.3f}"
    print(msg, flush=True)

p_te = to_um(predict(model, x_te, mean, std))
result = metrics(meta_te, y_te_um, p_te)
print(json.dumps(result, indent=2))
(RUN_DIR / "metrics.json").write_text(json.dumps(result, indent=2))
out = meta_te[["group", "part", "texture", "replica", "image_name", "azimuth", "polar"]].copy()
out["target_dft_um"], out["pred_dft_um"] = y_te_um, p_te
out = out.sort_values(["group", "pred_dft_um"])
out["predicted_rank"] = out.groupby("group").cumcount() + 1
out.to_csv(RUN_DIR / "ranked_test_predictions.csv", index=False)
torch.save({"state_dict": model.state_dict(), "channels": CHANNELS, "log_mu": mu, "log_sd": sd}, RUN_DIR / "checkpoint.pt")